In [1]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ['HADOOP_HOME'] = 'C:\\hadoop' 
#Start spark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("CartAbandonment") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
# Load parquet (NO header / inferSchema needed)
df = spark.read.parquet("../data/cleaned_ecommerce_logs.parquet")

In [4]:
# Split events
cart_df = df.filter(df.event_type == "cart")
purchase_df = df.filter(df.event_type == "purchase")


In [5]:
# Abandoned carts = in cart but NOT purchased
abandoned_df = cart_df.join(
    purchase_df,
    on=["user_id", "session_id", "product_id"],
    how="left_anti"
)
abandoned_df.show(5)

+-------+---------------+----------+-------------------+----------+-------+------------+--------------------+--------------------+-----------+---------+------+
|user_id|     session_id|product_id|          timestamp|event_type|  price|    referrer|       user_metadata|    product_metadata|   category|    brand|device|
+-------+---------------+----------+-------------------+----------+-------+------------+--------------------+--------------------+-----------+---------+------+
| User_0|SESS_0de6586001| ITEM_1345|2026-05-15 13:29:44|      cart|1288.25|facebook.com|{"device": "Table...|{"category": "Ele...|Electronics|    Apple|Tablet|
| User_0|SESS_2904635e7a| ITEM_4115|2026-05-03 17:38:15|      cart| 178.85|        NULL|{"device": "Table...|{"category": "Clo...|   Clothing|     Nike|Tablet|
| User_0|SESS_4ff5c95ed7| ITEM_4880|2026-01-14 17:47:27|      cart|  386.5|  newsletter|{"device": "Table...|{"category": "Boo...|      Books|Macmillan|Tablet|
| User_0|SESS_a1f7f6e73a| ITEM_1611|2026

In [ ]:
print(f"Total abandoned carts: {abandoned_df.count()}")

# Fix for Windows file renaming errors
spark.conf.set("mapreduce.fileoutputcommitter.algorithm.version", "2")

# Save output (coalesce to 1 file to prevent file lock crashes on Windows)
abandoned_df.coalesce(1).write.csv(
    "../data/abandonment_outputs",
    header=True,
    mode="overwrite"
)

spark.stop()


Total abandoned carts: 1421162
